**1. 데이터 다운**  
원본 데이터: https://www.kaggle.com/bittlingmayer/amazonreviews (Amazon 상품 리뷰 감성 분류, `__label__1` = 부정, `__label__2` = 긍정)

[수정] Kaggle은 로그인이 필요하므로, 같은 데이터를 제공하는 Hugging Face `amazon_polarity`에서 일부를 받아 원본과 같은 형식의 `train.txt`·`test.txt`로 저장한다. GloVe 임베딩도 함께 내려받는다.

> **수정본 안내 (2026-11 우병오)** — 현재 Colab(TensorFlow 2.16+ / Keras 3)에서 그대로 실행되도록 고친 버전이다. 모델 구조(윈도우 크기 2·3·4·5 필터 각 100개 + GloVe 임베딩)는 원본과 같다.
> - **데이터 자동 준비**: 원본은 Kaggle에서 직접 받은 `train.txt`·`test.txt`·`vocab.txt`를 업로드해야 했다 → 같은 Amazon 리뷰 데이터(amazon_polarity)를 Hugging Face에서 받아 일부(학습 20,000 / 평가 5,000)를 같은 형식(`__label__1`/`__label__2 문장`)으로 저장
> - **Keras 3 대응**: `keras.preprocessing.text.Tokenizer`(삭제됨) → 같은 동작의 간단한 토크나이저 함수, `keras.preprocessing.sequence.pad_sequences` → `keras.utils.pad_sequences`, `Embedding(weights=…, input_length=…)`(삭제된 인자) → `embeddings_initializer=Constant(…)`
> - **버그 수정**: 빈 줄 검사 `line.strip() == 0` → `not line.strip()`, 시각화 함수의 `history['acc']` → `history['accuracy']`(KeyError), 레이블을 `np.array`로 변환
> - NLTK `punkt`/`punkt_tab` 다운로드, GloVe 다운로드(스탠퍼드 → Hugging Face 미러) 셀 추가, 문장 최대 길이 256으로 제한(메모리·속도)
> - **런타임**: GPU(T4) 권장

In [ ]:
!pip install -q datasets
import os
from datasets import load_dataset

N_TRAIN, N_TEST = 20000, 5000          # 전체(360만/40만 문장)는 너무 크므로 일부만 사용
os.makedirs("/content", exist_ok=True)

def save_fasttext_format(split, n, path):
    # streaming=True: 전체(약 1GB)를 내려받지 않고 앞부분을 읽으며 섞어서 n개만 가져온다
    ds = load_dataset("fancyzhx/amazon_polarity", split=split, streaming=True).shuffle(seed=42, buffer_size=20000).take(n)
    with open(path, "w", encoding="utf-8") as fw:
        for ex in ds:
            text = (ex["title"] + ": " + ex["content"]).replace("\n", " ")
            fw.write("__label__%d %s\n" % (ex["label"] + 1, text))   # label 0 → __label__1(부정), 1 → __label__2(긍정)

if not os.path.exists("/content/train.txt"):
    save_fasttext_format("train", N_TRAIN, "/content/train.txt")
    save_fasttext_format("test", N_TEST, "/content/test.txt")
!head -n 3 /content/train.txt

In [ ]:
# GloVe 6B(약 822MB) — 스탠퍼드 서버가 응답하지 않으면 Hugging Face 미러에서 받는다.
if not os.path.exists("/content/glove.6B.50d.txt"):
    if not os.path.exists("/content/glove.6B.zip"):
        !wget -q --timeout=60 -O /content/glove.6B.zip http://nlp.stanford.edu/data/glove.6B.zip || wget -q -O /content/glove.6B.zip https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
    !unzip -q -o /content/glove.6B.zip glove.6B.50d.txt glove.6B.100d.txt -d /content
!ls -lh /content/glove.6B.*d.txt

**2. 패키지 선언**

In [ ]:
import os
import numpy as np
import nltk
import random
from collections import Counter
import keras
from keras.utils import pad_sequences      # [수정] keras.preprocessing.sequence → keras.utils (Keras 3)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)     # [수정] 최신 NLTK의 word_tokenize에 필요

**3. 학습 데이터와 평가 데이터**



In [ ]:
MAX_LEN_CAP = 256   # [수정] 아주 긴 리뷰 때문에 패딩 길이가 지나치게 커지지 않도록 제한

def read_dataset(dataset_type):
  max_seq_len = 0
  with open("/content/%s.txt" % dataset_type, "r", encoding="utf-8") as fr_handle:
    labels, sentences = [], []
    for line in fr_handle:
      if not line.strip():                      # [수정] 원본 line.strip() == 0 은 항상 False
        continue
      label = line.split(' ')[0]
      label = 0 if label == "__label__1" else 1 # 부정이면 0, 긍정이면 1

      sentence = ' '.join(line.split(' ')[1:]).strip()
      tokenized_sentence = nltk.word_tokenize(sentence)
      max_seq_len = max(max_seq_len, len(tokenized_sentence))

      labels.append(label)
      sentences.append(sentence)

    return np.array(labels), sentences, max_seq_len   # [수정] Keras 3 학습용으로 np.array 반환

TRAIN_LABELS, TRAIN_SENTENCES, TRAIN_MAX_SEQ_LEN = read_dataset("train")
TEST_LABELS, TEST_SENTENCES, TEST_MAX_SEQ_LEN = read_dataset("test")
MAX_SEQUENCE_LEN = min(max(TRAIN_MAX_SEQ_LEN, TEST_MAX_SEQ_LEN), MAX_LEN_CAP)

print("Train : ", len(TRAIN_SENTENCES))
for train_label, train_sent in zip(TRAIN_LABELS, TRAIN_SENTENCES[0:5]):
  print(train_label, ':', train_sent[:150])

print()
print("Test : ", len(TEST_SENTENCES))
for test_label, test_sent in zip(TEST_LABELS, TEST_SENTENCES[0:5]):
  print(test_label, ':', test_sent[:150])

print("MAX_SEQUENCE_LEN", MAX_SEQUENCE_LEN)

**4. 전처리 과정**  
1) Text를 tokenize하여 id 값으로 변경해 줍니다. (`texts_to_sequences`)  
2) id로 변경해준 문장들을 모두 문장 최대 길이로 padding 처리해 줍니다. (`pad_sequences`)

[수정] Keras 3에서 `keras.preprocessing.text.Tokenizer`가 삭제되어, 같은 역할(소문자화·구두점 제거·빈도순 번호 부여)을 하는 간단한 토크나이저를 직접 만들어 사용합니다. 원본의 `vocab.txt` 대신 학습 데이터에서 상위 `NUM_WORDS`개 단어를 사용합니다.

In [ ]:
import re

class SimpleTokenizer:
    """keras.preprocessing.text.Tokenizer(lower=True)와 같은 방식: 빈도순으로 1번부터 번호 부여, 0은 패딩"""
    FILTERS = re.compile(r'[!"#$%&()*+,\-./:;<=>?@\[\\\]^_`{|}~\t\n]')
    def __init__(self, num_words=None):
        self.num_words = num_words
        self.word_index = {}
    def _split(self, text):
        return self.FILTERS.sub(' ', text.lower()).split()
    def fit_on_texts(self, texts):
        counter = Counter(w for t in texts for w in self._split(t))
        self.word_index = {w: i + 1 for i, (w, _) in enumerate(counter.most_common())}
    def texts_to_sequences(self, texts):
        limit = self.num_words or (len(self.word_index) + 1)
        return [[self.word_index[w] for w in self._split(t)
                 if w in self.word_index and self.word_index[w] < limit] for t in texts]

NUM_WORDS = 20000
tokenizer = SimpleTokenizer(num_words=NUM_WORDS)
tokenizer.fit_on_texts(TRAIN_SENTENCES)
TRAIN_SEQUENCES = tokenizer.texts_to_sequences(TRAIN_SENTENCES)
TEST_SEQUENCES = tokenizer.texts_to_sequences(TEST_SENTENCES)
VOCAB_SIZE = NUM_WORDS          # 0(패딩) ~ NUM_WORDS-1 번만 사용

print(TRAIN_SENTENCES[2][:200])
print(TRAIN_SEQUENCES[2][:30])

X_train = pad_sequences(TRAIN_SEQUENCES, padding='post', truncating='post', maxlen=MAX_SEQUENCE_LEN)
X_test = pad_sequences(TEST_SEQUENCES, padding='post', truncating='post', maxlen=MAX_SEQUENCE_LEN)
print("PAD_SEQUENCES COMPLETES")
print(X_train.shape, X_test.shape)

**5. 모델 설정**  
1) Random으로 초기화된 임베딩이 아닌 pre-trained 된 GLoVE 임베딩으로 학습하고자 합니다.  
2) 따라서 학습 코퍼스에 있는 단어들 중 GLoVE 임베딩에 있는 단어들을 GLoVE 임베딩으로 초기화 해줍니다.  
3) 본 실험에서는 GLoVE 임베딩 크기가 50인 것과 100인 것을 통해 실험을 진행해 봅니다.  

In [ ]:
def create_embedding_matrix(filepath, word_index, embedding_dim, vocab_size):
    embedding_matrix = np.zeros((vocab_size, embedding_dim))   # GloVe에 없는 단어는 0 벡터
    found = 0
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            word, *vector = line.split()
            idx = word_index.get(word)
            if idx is not None and idx < vocab_size:
                embedding_matrix[idx] = np.array(vector, dtype=np.float32)[:embedding_dim]
                found += 1
    print(f"GloVe 벡터를 찾은 단어: {found}/{vocab_size - 1}")
    return embedding_matrix

EMBEDDING_DIM = 50   # 100으로 바꾸고 아래 파일명을 glove.6B.100d.txt로 바꿔 비교해 보자
embedding_matrix = create_embedding_matrix(
    '/content/glove.6B.%dd.txt' % EMBEDDING_DIM,
    tokenizer.word_index, EMBEDDING_DIM, VOCAB_SIZE)

**6. Accuracy와 Loss 시각화**  

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('ggplot')

def plot_history(history):
    acc = history.history['accuracy']   # [수정] metrics='accuracy'의 키 이름
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    x = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(x, acc, 'b', label='Training acc')
    plt.plot(x, val_acc, 'r', label='Validation acc')
    plt.title('Training and validation accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(x, loss, 'b', label='Training loss')
    plt.plot(x, val_loss, 'r', label='Validation loss')
    plt.title('Training and validation loss')
    plt.legend()

**7. CNN 모델 선언**  
1) Convolution 필터를 앞서 배운 것처럼 여러 개를 사용해 봅시다.  
2) 본 모델은 필터의 Window Size가 2, 3, 4, 5인 필터 각 100개씩을 사용해 모델을 학습합니다. 다양한 크기의 필터를 사용하면 성능이 더 올라갈까요?  
3) 정답은 "올라갑니다." 입니다. Convolution 필터가 보는 단어의 갯수가 다양하게 되기 때문에 문장의 local 정보와 global 정보 모두를 학습할 수 있게 됩니다. 그럼 실험을 통해 확인해 볼까요?  

In [ ]:
from keras import layers
from keras.models import Model

seq_input = layers.Input(shape=(MAX_SEQUENCE_LEN,), dtype='int32')
seq_embedded = layers.Embedding(VOCAB_SIZE,
                           EMBEDDING_DIM,
                           embeddings_initializer=keras.initializers.Constant(embedding_matrix),  # [수정] weights=, input_length= 인자 삭제(Keras 3)
                           trainable=True)(seq_input)

filters = [2, 3, 4, 5]
conv_models = []
for filter in filters:
  conv_feat = layers.Conv1D(filters=100,
                            kernel_size=filter,
                            activation='relu',
                            padding='valid')(seq_embedded)
  pooled_feat = layers.GlobalMaxPooling1D()(conv_feat)
  conv_models.append(pooled_feat)

conv_merged = layers.concatenate(conv_models, axis=1)

model_output = layers.Dropout(0.2)(conv_merged)
model_output = layers.Dense(10, activation='relu')(model_output)
logits = layers.Dense(1, activation='sigmoid')(model_output)

model = Model(seq_input, logits)
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

# 학습 시작
history = model.fit(X_train, TRAIN_LABELS,
                    epochs=10,
                    verbose=True,
                    validation_data=(X_test, TEST_LABELS),
                    batch_size=128)
# 결과 시각화
plot_history(history)